# Factualis: full pipeline on one T4

This notebook reproduces everything behind Factualis on a single free Colab T4 (16 GB):

1. A 7B teacher model, served with SGLang, generates training questions through CuratorKIT.
2. The teacher answers every question in our fixed answer format, and only answers that pass the word-for-word check are kept.
3. AlignTune fine-tunes Qwen2.5-3B, first with SFT and then with DPO.
4. Every model is evaluated on the 94 hand-checked test questions, and the website is started at the end.

Before you run it, switch the runtime to a T4 GPU and upload `factualis_colab.zip` to `MyDrive/Factualis/`. You can build the zip from the repository with `python -m tools.package_for_colab`.

A full run takes roughly 6 to 8 hours. Every step saves its output to Drive (`MyDrive/Factualis/runs/<run name>/`) and is skipped if that output already exists, so if Colab disconnects you can just run all the cells again from the top.

Two documents, the SEBI insider trading regulations and the bank's Employee Personal Dealing Policy, are never shown to the model during training. The evaluation reports them separately to see how well the model handles policy it has never seen.

### Settings

`PILOT = True` pushes a small slice of data through every step in about an hour and a half. It is worth doing once to catch problems before committing to the full run. The rest are the sizes we used: 260 clauses for curation, two grounded questions and one trap question per clause, two epochs of SFT and one of DPO.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/Factualis"
PILOT = False
RUN_NAME = "run1" + ("-pilot" if PILOT else "")

TEACHER = "Qwen/Qwen2.5-7B-Instruct-AWQ"
BASE = "Qwen/Qwen2.5-3B-Instruct"
HOLDOUT = ["sebi_pit_regulations", "fb_personal_dealing_policy"]

CURATE_CLAUSES = 40 if PILOT else 260
QA_PER_CLAUSE = 2
TRAPS_PER_CLAUSE = 1
LABEL_LIMIT = 80 if PILOT else 0
SFT_EPOCHS = 1 if PILOT else 2
DPO_EPOCHS = 1
SAVE_MERGED_TO_DRIVE = False
SGLANG_SPEC = "sglang[all]"

### Mount Drive and unpack the code

The project zip is unpacked into `/content/Factualis`. Its `output/` folder is a link to the run folder on Drive, so everything the pipeline writes survives a disconnect.

In [ ]:
import os, json, re, shutil, subprocess, time, zipfile, pathlib, urllib.request
from google.colab import drive
drive.mount("/content/drive")

gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", gpu)
assert "T4" in gpu, "Switch the runtime to a T4 GPU (Runtime > Change runtime type)."

PROJ = "/content/Factualis"
RUN_DIR = f"{DRIVE_DIR}/runs/{RUN_NAME}"
CKPT = "/content/checkpoints"
DRIVE_CKPT = f"{RUN_DIR}/checkpoints"
for d in (RUN_DIR, f"{RUN_DIR}/output", DRIVE_CKPT, CKPT):
    os.makedirs(d, exist_ok=True)

zip_path = f"{DRIVE_DIR}/factualis_colab.zip"
assert os.path.exists(zip_path), f"Upload factualis_colab.zip to {DRIVE_DIR} first."
shutil.rmtree(PROJ, ignore_errors=True)
zipfile.ZipFile(zip_path).extractall(PROJ)
shutil.rmtree(f"{PROJ}/output", ignore_errors=True)
os.symlink(f"{RUN_DIR}/output", f"{PROJ}/output")
os.chdir(PROJ)
print("code unpacked to", PROJ, "| results go to", RUN_DIR)

### Install the three environments

SGLang, CuratorKIT/AuditKIT and AlignTune each want a different PyTorch build, so they go into three separate virtual environments, installed in parallel with `uv`. This takes about ten minutes the first time in a session.

In [ ]:
ENVS = {
    "serve": f'"{SGLANG_SPEC}" openai ninja',
    "app": '"curatorkit[all]" auditkit pydantic openai rank_bm25 pypdf fastapi uvicorn',
    "train": "aligntune unsloth peft transformers",
}

subprocess.run("pip -q install uv", shell=True, check=True)
procs = {}
for name, pkgs in ENVS.items():
    root = f"/content/envs/{name}"
    if os.path.exists(f"{root}/.ok"):
        continue
    cmd = f"uv venv -q --python 3.11 {root} && uv pip install -q --python {root}/bin/python {pkgs} && touch {root}/.ok"
    procs[name] = subprocess.Popen(cmd, shell=True, stdout=open(f"/content/install_{name}.log", "w"), stderr=subprocess.STDOUT)
for name, p in procs.items():
    rc = p.wait()
    print(f"{name}: {'ok' if rc == 0 else 'failed, see /content/install_' + name + '.log'}")
    if rc:
        print(open(f"/content/install_{name}.log").read()[-3000:])

PY = {k: f"/content/envs/{k}/bin/python" for k in ENVS}
BIN = {k: f"/content/envs/{k}/bin" for k in ENVS}
for k in ENVS:
    assert os.path.exists(f"/content/envs/{k}/.ok"), f"environment {k} did not install"
print("environments ready")

### Helper functions

A few small functions the rest of the notebook uses:

- `sh` runs a command inside the unpacked project and streams its output.
- `start_sglang` / `stop_sglang` start and stop the model server. Only one model fits on the T4 at a time, so every step that needs a different model stops the old server first. The server is started in its own process session, otherwise pressing Colab's stop button on any cell would also kill it.
- `open_website` starts the FastAPI app and a free Cloudflare tunnel, then prints the public link.

The `pkill` patterns are written as `web[.]app` and `sglang[.]launch_server` on purpose. A plain `web.app` also matches Colab's own `/datalab/web/app.js` process, and killing that disconnects the runtime.

In [ ]:
def sh(cmd, env=None):
    print(f"$ {cmd}")
    p = subprocess.Popen(cmd, shell=True, cwd=PROJ, env={**os.environ, **(env or {})},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"command failed: {cmd}")


def done(path):
    return os.path.exists(path)


def gpu_used_mb():
    out = subprocess.run("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits",
                         shell=True, capture_output=True, text=True).stdout
    return int(out.strip().splitlines()[0])


def stop_sglang():
    subprocess.run("pkill -f 'sglang[.]launch_server'", shell=True)
    for i in range(60):
        if gpu_used_mb() < 1000:
            break
        if i == 20:
            subprocess.run("pkill -9 -f 'sglang[.]launch_server'", shell=True)
        time.sleep(2)
    print(f"SGLang stopped, GPU memory in use: {gpu_used_mb()} MiB")


def start_sglang(model, port=30000, mem=0.80, extra="", timeout=1200):
    stop_sglang()
    flags = ("--dtype half --attention-backend triton --sampling-backend pytorch --grammar-backend xgrammar "
             "--context-length 8192 --served-model-name default --host 0.0.0.0 --enable-metrics")
    if "AWQ" in model:
        flags += " --quantization awq"
    log = f"/content/sglang_{port}.log"
    cmd = f"{PY['serve']} -m sglang.launch_server --model-path {model} --port {port} --mem-fraction-static {mem} {flags} {extra}"
    print(f"$ {cmd}")
    env = {**os.environ, "PATH": f"{BIN['serve']}:/usr/local/cuda/bin:{os.environ['PATH']}", "CUDA_HOME": "/usr/local/cuda"}
    p = subprocess.Popen(cmd, shell=True, cwd=PROJ, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            urllib.request.urlopen(f"http://localhost:{port}/health", timeout=30)
            print(f"SGLang ready after {time.time() - t0:.0f}s ({model})")
            return
        except Exception:
            if p.poll() is not None:
                print(open(log).read()[-3000:])
                raise RuntimeError(f"SGLang exited, full log in {log}")
            time.sleep(5)
    raise TimeoutError(f"SGLang not ready after {timeout}s, see {log}")


def open_website(extra_args=""):
    subprocess.run("pkill -f 'm web[.]app --port'; pkill -f 'cloudflared tunnel'", shell=True)
    time.sleep(1)
    subprocess.Popen(f"{PY['app']} -u -m web.app --port 8000 --aligned-port 30000 {extra_args}", shell=True, cwd=PROJ,
                     stdout=open("/content/web.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        try:
            urllib.request.urlopen("http://localhost:8000/api/examples", timeout=3)
            break
        except Exception:
            time.sleep(2)
    else:
        print(open("/content/web.log").read()[-2000:])
        return
    if not done("/content/cloudflared"):
        sh("wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared")
    subprocess.Popen("/content/cloudflared tunnel --no-autoupdate --url http://localhost:8000", shell=True,
                     stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        time.sleep(2)
        m = re.search(r"https://[\w-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
        if m:
            print("\nFactualis is live at:", m.group(0))
            return
    print(open("/content/tunnel.log").read()[-1500:])


TRAIN_ENV = {"CKPT_DIR": CKPT, "BASE_MODEL": BASE, "SFT_EPOCHS": str(SFT_EPOCHS), "DPO_EPOCHS": str(DPO_EPOCHS)}
HOLDOUT_ARGS = " ".join(f"--holdout-doc {d}" for d in HOLDOUT)

## Step 1. Start the teacher model

The teacher is Qwen2.5-7B-Instruct in 4-bit AWQ, which fits on the T4 next to its KV cache. The quick check sends two questions with the answer format enforced by XGrammar, one answerable and one that should be refused, and prints the result.

In [ ]:
if not done("output/fincomp/funnel.json"):
    start_sglang(TEACHER)
    sh(f"{PY['app']} -m pipeline.check_sglang_server --port 30000")
else:
    print("training data already built for this run, skipping the teacher")

## Step 2. Generate questions with CuratorKIT

We take clauses from the training documents only (the bank's own procedures first, then a shuffled sample of the regulations) and run two CuratorKIT pipelines over them:

- `configs/curatorkit_grounded_qa.yaml` writes questions that the clause answers, with CuratorKIT's hallucination and reward filters on.
- `configs/curatorkit_near_miss_traps.yaml` writes questions that sound related but that the clause does not answer. These teach the model to refuse.

`--async` sends 16 requests at a time; without it CuratorKIT waits for each answer before sending the next.

In [ ]:
import random

clauses = [json.loads(l) for l in open("data/policy_clauses.jsonl", encoding="utf-8")]
train_clauses = [c for c in clauses if c["doc_id"] not in HOLDOUT]
procedures = [c for c in train_clauses if c["doc_id"].startswith("fb_")]
regulations = [c for c in train_clauses if not c["doc_id"].startswith("fb_")]
random.Random(13).shuffle(regulations)
selected = (procedures + regulations)[:CURATE_CLAUSES]
with open("data/policy_clauses_for_curation.jsonl", "w", encoding="utf-8") as f:
    for c in selected:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
print(f"{len(selected)} clauses selected, {sum(c['doc_id'].startswith('fb_') for c in selected)} of them from bank procedures")


def curated(out_dir):
    if not done(f"{out_dir}/manifest.json"):
        return False
    counts = json.load(open(f"{out_dir}/manifest.json")).get("stage_counts", {})
    return any("Task" in k and v.get("output_count", 0) > 0 for k, v in counts.items())


os.makedirs("output/configs", exist_ok=True)
pipelines = {"qa": ("curatorkit_grounded_qa", QA_PER_CLAUSE), "traps": ("curatorkit_near_miss_traps", TRAPS_PER_CLAUSE)}
for name, (config, n) in pipelines.items():
    cfg = open(f"configs/{config}.yaml").read().replace("data/policy_clauses.jsonl", "data/policy_clauses_for_curation.jsonl")
    cfg = re.sub(r"num_questions: \d+", f"num_questions: {n}", cfg)
    path = f"output/configs/{config}.yaml"
    open(path, "w").write(cfg)
    out = f"output/curator_{name}"
    if curated(out):
        print(f"{name}: already done")
        continue
    shutil.rmtree(out, ignore_errors=True)
    sh(f"{BIN['app']}/curatorkit run {path} --dry-run")
    sh(f"{BIN['app']}/curatorkit run {path} --output-dir {out} --async")

for name in pipelines:
    m = json.load(open(f"output/curator_{name}/manifest.json"))
    print(f"\nCuratorKIT manifest, {name}:")
    print(json.dumps({k: v for k, v in m.items() if k != "config"}, indent=1)[:1500])

## Step 3. Label and filter the training data

The teacher now answers every question (CuratorKIT's plus our hand-written seed questions) in the full answer format, over the same six clauses the student model will see. An answer is kept only if every quote is copied exactly from the clauses, every figure appears in them, and the status matches the kind of question.

The kept answers become the SFT data. For DPO, each kept answer is paired with a copy that has exactly one thing broken: a wrong threshold, a dropped condition, a reworded quote, a refusal of an answerable question, or an answer to a trap. `funnel.json` records how many answers were kept and why the rest were rejected.

In [ ]:
if not done("output/fincomp/funnel.json"):
    sh(f"{PY['app']} -m pipeline.build_training_data {HOLDOUT_ARGS} --limit {LABEL_LIMIT} --workers 16")
print(open("output/fincomp/funnel.json").read())

## Step 4. Evaluate the untrained model

This gives us the baseline. Run A is plain prompting, run B is the same model with the answer format enforced by XGrammar. We also measure how much RadixAttention's prefix cache speeds up the first token when several questions share the same policy text.

In [ ]:
if not (done("output/eval/A_base_prompted.json") and done("output/eval/B_base_xgrammar.json")):
    start_sglang(BASE)
    if not done("output/eval/A_base_prompted.json"):
        sh(f"{PY['app']} -m pipeline.evaluate --name A_base_prompted --unconstrained {HOLDOUT_ARGS}")
    if not done("output/eval/B_base_xgrammar.json"):
        sh(f"{PY['app']} -m pipeline.evaluate --name B_base_xgrammar {HOLDOUT_ARGS}")
    if not done("output/eval/radix_bench.json"):
        sh(f"{PY['app']} -m pipeline.benchmark_prefix_cache --port 30000")
sh(f"{PY['app']} -m pipeline.evaluate --report")

## Step 5. Supervised fine-tuning

SGLang has to be stopped first, because the T4 cannot hold the server's memory pool and a training run at the same time. Training uses AlignTune with the Unsloth backend: 4-bit base weights, LoRA adapters, fp16 (the T4 has no bf16) and gradient checkpointing.

The LoRA adapter is copied to Drive as soon as training finishes. If Colab disconnects later, the next run only has to merge it again, which takes about a minute on the GPU.

In [ ]:
stop_sglang()
assert gpu_used_mb() < 1500, "GPU is still busy. Restart the session (Runtime > Restart session) and run again from the top."

sft_local, sft_merged = f"{CKPT}/qwen3b-fincomp-sft", f"{CKPT}/qwen3b-fincomp-sft-merged"
sft_drive = f"{DRIVE_CKPT}/qwen3b-fincomp-sft"
if not done(f"{sft_merged}/config.json"):
    if not done(f"{sft_drive}/adapter_config.json"):
        if not done(f"{sft_local}/adapter_config.json"):
            sh(f"{PY['train']} -m pipeline.train sft", env=TRAIN_ENV)
        shutil.copytree(sft_local, sft_drive, dirs_exist_ok=True)
    sh(f"{PY['train']} -m pipeline.train merge --adapter {sft_drive} --base {BASE} --out {sft_merged}", env=TRAIN_ENV)
print("SFT model:", sft_merged)

## Step 6. DPO

DPO starts from the merged SFT model and learns to prefer the verified answer over its broken copy in each pair.

In [ ]:
dpo_local, dpo_merged = f"{CKPT}/qwen3b-fincomp-dpo", f"{CKPT}/qwen3b-fincomp-dpo-merged"
dpo_drive = f"{DRIVE_CKPT}/qwen3b-fincomp-dpo"
if not done(f"{dpo_merged}/config.json"):
    if not done(f"{dpo_drive}/adapter_config.json"):
        if not done(f"{dpo_local}/adapter_config.json"):
            sh(f"{PY['train']} -m pipeline.train dpo", env=TRAIN_ENV)
        shutil.copytree(dpo_local, dpo_drive, dirs_exist_ok=True)
    sh(f"{PY['train']} -m pipeline.train merge --adapter {dpo_drive} --base {sft_merged} --out {dpo_merged}", env=TRAIN_ENV)
if SAVE_MERGED_TO_DRIVE:
    shutil.copytree(dpo_merged, f"{DRIVE_CKPT}/qwen3b-fincomp-dpo-merged", dirs_exist_ok=True)
print("DPO model:", dpo_merged)

## Step 7. Evaluate the fine-tuned models

Run C is the SFT model and run D the SFT + DPO model, both with XGrammar. Run E is the DPO model without XGrammar, to see whether training alone teaches it the format. The last cell of this step prints the table comparing all five runs.

In [ ]:
for name, model in (("C_sft_xgrammar", sft_merged), ("D_aligned_xgrammar", dpo_merged)):
    if not done(f"output/eval/{name}.json"):
        start_sglang(model)
        sh(f"{PY['app']} -m pipeline.evaluate --name {name} {HOLDOUT_ARGS}")
        if name.startswith("D"):
            sh(f"{PY['app']} -m pipeline.evaluate --name E_aligned_unconstrained --unconstrained {HOLDOUT_ARGS}")
            sh(f"{PY['app']} -m pipeline.benchmark_prefix_cache --port 30000 --out output/eval/radix_bench_aligned.json")
sh(f"{PY['app']} -m pipeline.evaluate --report")

from IPython.display import Markdown, display
display(Markdown(open("output/eval/report.md").read()))
for f in ("radix_bench.json", "radix_bench_aligned.json"):
    if done(f"output/eval/{f}"):
        print(f, open(f"output/eval/{f}").read())

## Step 8. Open the website

Two models do not fit on the T4 together, so the untrained model's answers to the four example questions are computed once and saved, and the page shows them under "Before training". Then the fine-tuned model is served and the website gets a public `trycloudflare.com` link. The link changes every time this cell runs and stops working when the session ends.

In [ ]:
examples_cache = "output/demo_base_cache.json"
if not done(examples_cache):
    start_sglang(BASE)
    sh(f"{PY['app']} -m web.app --precompute-examples {examples_cache}")
start_sglang(dpo_merged)
open_website(f"--base-cache {examples_cache}")

### What ended up on Drive

In [ ]:
for p in sorted(pathlib.Path(RUN_DIR).rglob("*")):
    if p.is_file() and p.suffix in (".json", ".md") and "configs" not in p.parts:
        print(p.relative_to(RUN_DIR), f"{p.stat().st_size / 1e3:.1f} KB")